# Variable coefficients

The implicit step of an energy-conserving particle-in-cell scheme solves, in each nonlinear
iteration, an elliptic equation whose coefficient is the permittivity plus the plasma's
response over the step,

$$
-\nabla\cdot(\mathsf{K}\,\nabla\phi) = \rho, \qquad \mathsf{K} = \varepsilon_0 + \Delta t\,\chi,
$$

with $\chi$ the susceptibility from the markers' moments: large where the plasma is dense,
a tensor where it is magnetized. `NodalPoisson` assembles this operator on the composite nodes
(`coefficient=` per leaf cell, a number, a diagonal or a symmetric tensor; `edge_coefficient=`
per edge), and the `"multigrid"` backend solves it with a number of iterations that does not
grow with the refinement levels or the cell count.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from patchwork import AMRHierarchy, MultigridBackend, NodalPoisson, plot_patches

## A dense, magnetized blob on a refined mesh

A blob of plasma in the middle of the box, refined twice. Inside it $\Delta t\,\chi$ is a hundred
times $\varepsilon_0$ across the field lines and a little less along them, at an angle: a full
tensor per cell. The coefficient is given per leaf cell, in the order of `solver.nodes`,
evaluated at `nodes.cell_centers`.

In [ ]:
mesh = AMRHierarchy(base_shape=(32, 32))
mesh.add_patch(level=1, box=((16, 48), (16, 48)))
mesh.add_patch(level=2, box=((48, 80), (48, 80)))
solver = NodalPoisson(mesh, backend="multigrid")
nodes = solver.nodes

x, y = nodes.cell_centers.T
density = np.exp(-((x - 0.5) ** 2 + (y - 0.5) ** 2) / 0.02)

angle = np.pi / 6
along = np.array([np.cos(angle), np.sin(angle)])
across = np.array([-np.sin(angle), np.cos(angle)])
response = 100.0 * density
tensor = (
    np.eye(2)  # epsilon_0 = 1
    + (0.5 * response)[:, None, None] * np.outer(along, along)
    + response[:, None, None] * np.outer(across, across)
)
solver.set_coefficient(tensor)

px, py = nodes.coordinates.T
charge = np.exp(-((px - 0.4) ** 2 + (py - 0.6) ** 2) / 0.002)
potential = solver.solve(charge)
print(
    f"{solver.free.size} unknowns, {solver.solver.num_levels} multigrid levels, "
    f"{solver.solver.iterations} CG iterations"
)

Where the response is large the potential is flat: the plasma screens the field of the charge
(left of the centre) over the whole blob.

In [ ]:
grid = nodes.to_level(2) @ potential
shape = nodes.level_shape(2)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
image = axes[0].imshow(grid.reshape(shape).T, origin="lower", extent=(0, 1, 0, 1))
fig.colorbar(image, ax=axes[0], label=r"$\phi$")
axes[0].set_title("potential")
plot_patches(mesh, ax=axes[1])
axes[1].set_title("patches")
fig.tight_layout()

## Iterations

The multigrid's levels are the hierarchy without its finest level, without the next, down to
the base level, then the base level coarsened by two along each axis
(`patchwork.solvers.multigrid`); the coarse operators are Galerkin products, so the
coefficient needs nothing of its own on them. The number of conjugate-gradient iterations to a
relative residual of $10^{-10}$ stays the same as patches and cells are added:

In [ ]:
def nested(n, levels):
    mesh = AMRHierarchy(base_shape=(n, n))
    low, high = n // 4, 3 * n // 4
    for level in range(1, levels + 1):
        width = high - low
        low, high = 2 * low + width // 2, 2 * low + 3 * width // 2
        low, high = low - low % 2, high - high % 2
        mesh.add_patch(level=level, box=((low, high), (low, high)))
    return mesh


print("cells  levels  unknowns  iterations")
for n in (32, 64, 128):
    for levels in (0, 1, 2, 3):
        solver = NodalPoisson(nested(n, levels), backend="multigrid")
        nodes = solver.nodes
        x, y = nodes.cell_centers.T
        solver.set_coefficient(
            1.0 + 100.0 * np.exp(-((x - 0.5) ** 2 + (y - 0.5) ** 2) / 0.02)
        )
        px, py = nodes.coordinates.T
        solver.solve(np.sin(np.pi * px) * np.sin(2 * np.pi * py))
        print(
            f"{n:5d}  {levels:6d}  {solver.free.size:8d}  {solver.solver.iterations:10d}"
        )

## Inside a nonlinear iteration

A new coefficient in every step reassembles the operator and the Galerkin products, and keeps
the interpolations between the levels (`set_coefficient`). One V-cycle alone (`solver.solver.cycle`,
or `as_linear_operator()` for SciPy) is the preconditioner of an outer Krylov method, such as the
Jacobian-free Newton–Krylov iteration of the implicit step.

In [ ]:
import scipy.sparse.linalg

solver = NodalPoisson(nested(64, 2), backend=MultigridBackend(rtol=1e-8))
interpolations = solver.solver.interpolations
for step in range(3):
    solver.set_coefficient(1.0 + 10.0 * step)
    assert solver.solver.interpolations is interpolations

right = solver.load @ np.ones(solver.nodes.num_nodes)
counts = []
x, info = scipy.sparse.linalg.cg(
    solver.matrix,
    right,
    rtol=1e-8,
    M=solver.solver.as_linear_operator(),
    callback=lambda _: counts.append(1),
)
print(f"SciPy CG with one V-cycle as preconditioner: {len(counts)} iterations")